# Antarctic Soundscape Sanity Check Demo

This notebook demonstrates how the `soundsanity` library detects recording defects. Since the provided example recordings are clean, we will use the library's degradation utilities to simulate the following audio issues:
1. **Too many clicks** (e.g. from electrical interface noise)
2. **Clipped/Saturated signals** (e.g. from input gain set too high)
3. **Failed recordings / Silence** (e.g. from unplugged mics or cable failures)
4. **Too noisy recordings** (e.g. from wind or bad preamps)

We will run the analysis functions and visualize the results.

### 1. Imports and setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import essentia.standard as es
import soundsanity as ss

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.style.use("seaborn-v0_8-whitegrid")

### 2. Loading a clean reference file

We will load `nosotras_descansamos.wav` (which translates to "we rest", a quiet environmental recording) as our clean reference signal.

In [ ]:
ref_file = "../audio_examples/nosotras_descansamos.wav"
sample_rate = 44100
clean_audio = es.MonoLoader(filename=ref_file, sampleRate=sample_rate)()

# Run a baseline check to verify it is flagged clean
clean_report = ss.analyze_recording(ref_file)
print(f"Clean File Status: {clean_report['status']}")
print(f"Clicks count: {clean_report['clicks']['clicks_count']}")
print(f"Saturation %: {clean_report['saturation']['saturation_ratio'] * 100:.4f}%")
print(f"Silence ratio: {clean_report['silence']['silence_ratio']:.2%}")

### 3. Click Detection Demo

We will inject clicks using `add_clicks` and verify that the clicks are counted and flagged.

In [ ]:
# Inject 80 clicks (approx 0.84 clicks/sec over 95s)
clicky_audio, true_click_times = ss.add_clicks(clean_audio, num_clicks=80, click_amplitude=0.9, sample_rate=sample_rate)

# Run detection with a lower click rate threshold (e.g. 0.5 clicks/sec) to trigger it
custom_config = {"max_clicks_rate": 0.5}
click_res = ss.detect_clicks(clicky_audio, sample_rate, config=custom_config)

print(f"Injected clicks: {len(true_click_times)}")
print(f"Detected clicks: {click_res['clicks_count']} (Rate: {click_res['clicks_rate']:.2f} clicks/sec)")
print(f"Is flagged as clicky? {click_res['is_clicky']}")

# Let's visualize a 5-second segment containing some detected clicks
fig, ax = ss.plot_waveform_with_issues(clicky_audio[:sample_rate * 5], sample_rate,
                                      click_timestamps=[t for t in click_res['click_timestamps'] if t < 5.0],
                                      title="Injected Clicks in Waveform (First 5 seconds)")

### 4. Saturation / Clipping Detection Demo

We will clip the signal using `add_clipping` and verify that `detect_saturation` flags it.

In [ ]:
# Simulate severe clipping
clipped_audio = ss.add_clipping(clean_audio, clip_threshold=0.005)

sat_res = ss.detect_saturation(clipped_audio, sample_rate)

print(f"Saturation Ratio: {sat_res['saturation_ratio'] * 100:.3f}%")
print(f"Saturation duration: {sat_res['saturation_duration']:.2f}s")
print(f"Is flagged as saturated? {sat_res['is_saturated']}")

# Let's plot a close-up of a few milliseconds of the clipped signal
plt.figure(figsize=(10, 3))
plt.plot(np.arange(500) / sample_rate * 1000, clipped_audio[10000:10500], color='#e74c3c')
plt.title("Zoomed Saturated Signal (Flat-tops)")
plt.xlabel("Time (milliseconds)")
plt.ylabel("Amplitude")
plt.ylim(-1.1, 1.1)
plt.show()

### 5. Silence / Recording Failure Demo

We will attenuate the signal to simulate an empty channel.

In [ ]:
# Attenuate to -95 dB (essentially silent)
silent_audio = ss.make_silent(clean_audio, level_db=-95)

silence_res = ss.detect_silence(silent_audio, sample_rate)

print(f"Silent frames ratio: {silence_res['silence_ratio']:.2%}")
print(f"Is flagged as silent (recording failure)? {silence_res['is_silent']}")

### 6. High Noise Floor / SNR Demo

We will add white noise to simulate high-wind or bad gain structure.

In [ ]:
# Add noise to get a very poor SNR of 3 dB
noisy_audio = ss.add_noise(clean_audio, target_snr_db=3)

noise_res = ss.estimate_noise(noisy_audio, sample_rate)

print(f"Overall RMS level: {noise_res['rms_db']:.1f} dB")
print(f"Estimated Noise Floor: {noise_res['noise_floor_db']:.1f} dB")
print(f"Estimated SNR index: {noise_res['snr_db']:.1f} dB")
print(f"Is flagged as noisy? {noise_res['is_noisy']}")

### 7. Full Diagnostic Report & Visualization

Let's run `analyze_recording` on a degraded audio signal containing multiple issues (clicks + noise) and show the diagnostic plot.

In [ ]:
# Create a combined problem audio: clicky + noisy
bad_audio, clicks = ss.add_clicks(clean_audio, num_clicks=150, click_amplitude=0.85, sample_rate=sample_rate)
bad_audio = ss.add_noise(bad_audio, target_snr_db=8)

# Save to temporary WAV for analyze_recording
temp_output_path = "../audio_examples/temp_degraded_demo.wav"
es.MonoWriter(filename=temp_output_path, sampleRate=sample_rate)(bad_audio)

try:
    report = ss.analyze_recording(temp_output_path, config={"max_clicks_rate": 0.5})
    print(f"Combined Bad Audio Status: {report['status']}")
    print(f"Issues Detected:\n  * " + "\n  * ".join(report['issues']))
    
    # Plot quality report
    fig, axs = ss.plot_quality_report(bad_audio, sample_rate, report, title="Combined Degradation Diagnostic Report")
finally:
    # Clean up
    if os.path.exists(temp_output_path):
        os.remove(temp_output_path)